# CIFRADO DE UNA SOLA VIA
## Explicacion General de todo el codigo
Este codigo actua como una licuadora criptografica. Su objetivo es tomar cualquier texto (una contraseña, un mensaje o un libro entero) y triturarlo hasta convertirlo en una "huella digital" de tamaño fijo (32 ceros y unos, que luego se muestran como 8 letras/numeros).

## Explicacion paso a paso de las funciones

### 1. Traductores (De texto a maquina y viceversa)
*   **texto_a_bits(texto)**: La computadora no lee letras. Esta funcion toma el texto y convierte cada caracter en 8 bits (ceros y unos) usando la tabla ASCII.
*   **bits_a_hex(bits_str)**: Leer 32 ceros y unos seguidos es confuso para un humano. Esta funcion agrupa esos bits y los traduce a Hexadecimal (numeros del 0 al 9 y letras de la A a la F), haciendolo mucho mas corto de leer.

### 2. Las Herramientas de Mezcla
*   **rotar_izq(cadena, n)**: Es como un carrusel. Toma los primeros n bits de la izquierda, los arranca y los pega al final (a la derecha). Desordena las posiciones sin borrar nada.
*   **aplicar_xor(bits_a, bits_b)**: Compara dos filas de bits. Si los bits son diferentes (un 1 y un 0), anota un 1. Si son iguales, anota un 0. Es el "interruptor" de caos.
*   **aplicar_and(bits_a, bits_b)**: La destructora. Compara dos filas de bits. Solo si ambos son 1, anota un 1. Si hay algun 0, anota un 0. Aqui se pierde informacion para siempre, haciendo que el codigo sea irreversible.

### 3. (comprimir_irreversible)
Esta es la funcion que hace todo el trabajo pesado:
1.  **Relleno (Padding):** Obliga a que la cadena de ceros y unos de tu texto sea un multiplo exacto de 32. Si faltan pedazos, le inyecta ceros y unos al final.
2.  **Estado y Mascara:** Prepara dos herramientas constantes: un estado (la base de nuestra mezcla) y una mascara (el molde con el que vamos a golpear la mezcla).
3.  **El Bucle (La Licuadora):** Corta el texto en bloques de 32 bits. Toma el primer bloque, lo choca contra el estado (aplicando rotaciones, XOR y AND). El caos resultante se convierte en el nuevo estado. Luego repite esto con el siguiente bloque de tu texto, hasta que no quede nada.
4.  Retorna el ultimo estado de 32 bits que quedo en la maquina.

---

## Ejemplo conl a palabra hola

### Paso 1: Traduccion
La palabra "hola" tiene exactamente 4 letras. Como cada letra pesa 8 bits, se tiene:
*   4 letras * 8 = 32 bits exactos.
*   El codigo traduce "hola" a esto: 01101000011011110110110001100001

### Paso 2: El Relleno (Padding)
El programa revisa si el texto es multiplo de 32. Como "hola" mide exactamente 32 bits, no le agrega ningun relleno. Pasa directo a la licuadora.

### Paso 3: El Choque (Una sola ronda)
Como "hola" solo tiene un bloque de 32 bits, el bucle de la maquina se ejecuta una sola vez:
1.  El programa toma los 32 bits de "hola".
2.  Toma el estado inicial (que siempre es el mismo al arrancar) y lo rota 5 posiciones.
3.  Aplica la destructora (AND) entre el estado rotado y los bits de "hola". Si ambos no son 1, los vuelve 0.
4.  Luego hace dos cruces de caos (XOR) para revolver mas los bits.
5.  Finalmente, choca ese resultado contra la mascara fija usando otro XOR.

### Paso 4: El Resultado Final
Al terminar esa unica ronda, el bloque de la palabra "hola" original ya no existe; se transformo en una nueva cadena incomprensible de 32 bits.
Esa cadena sale de la funcion y pasa por bits_a_hex, entregando un codigo cortito af9d9ede que representa irreversiblemente la palabra "hola".

In [1]:
def texto_a_bits(texto):
    return ''.join(f"{ord(c):08b}" for c in texto)

def rotar_izq(cadena, n):
    if not cadena: return cadena
    n = n % len(cadena)
    return cadena[n:] + cadena[:n]

def aplicar_xor(bits_a, bits_b):
    return ''.join('1' if a != b else '0' for a, b in zip(bits_a, bits_b))

def aplicar_and(bits_a, bits_b):
    return ''.join('1' if a == '1' and b == '1' else '0' for a, b in zip(bits_a, bits_b))

def bits_a_hex(bits_str):
    return f"{int(bits_str, 2):0{len(bits_str) // 4}x}"

def comprimir_irreversible(entrada):
    bits = texto_a_bits(entrada)
    tamaño_bloque = 32

    while len(bits) % tamaño_bloque != 0:
        bits += '1' if len(bits) % 2 == 0 else '0'

    estado = '10100101110000110101101011110000'
    mascara = '00111100110000111100110010101010'

    for i in range(0, len(bits), tamaño_bloque):
        bloque = bits[i:i+tamaño_bloque]
        estado_rotado = rotar_izq(estado, 5)
        mezcla_and = aplicar_and(estado_rotado, bloque)
        paso1 = aplicar_xor(estado_rotado, bloque)
        paso2 = aplicar_xor(paso1, rotar_izq(mezcla_and, 3))
        estado = aplicar_xor(paso2, mascara)
    return estado

entrada = input("Ingresa la palabra o clave: ")
if not entrada:
    entrada = "a"

enc_entrada_b = comprimir_irreversible(entrada)
enc_entrada_h = bits_a_hex(enc_entrada_b)

print("\n" + "=" * 55)
print(f"Texto original        : {entrada}")
print(f"Longitud de entrada   : {len(texto_a_bits(entrada))} bits")
print("-" * 55)
print(f"Resultado (bin) : {enc_entrada_b}")
print(f"Longitud de salida    : {len(enc_entrada_b)} bits")
print(f"Resultado (hex) : {enc_entrada_h}")
print("=" * 55)

Ingresa la palabra o clave: hola

Texto original        : hola
Longitud de entrada   : 32 bits
-------------------------------------------------------
Resultado (bin) : 10101111100111011001111011011110
Longitud de salida    : 32 bits
Resultado (hex) : af9d9ede
